# Level 3: trends, with functime and polars-ds

Level 2 made monthly series. This level asks which of them trend and by how much, honestly: a
series of a dozen noisy points has a slope by construction, the question is whether the slope is
larger than the noise. Three tools, all polars expressions that run inside `group_by().agg()`:

- [polars-ds](https://polars-ds-extension.readthedocs.io/) `lin_reg_report`: the slope with its
  standard error, p-value and confidence interval. DuckDB's `regr_slope` is the SQL twin of the
  slope.
- [functime](https://docs.functime.ai/) feature extractors (`.ts`): `linear_trend`,
  `mean_change`, `absolute_sum_of_changes`, `longest_streak_above_mean`, one number per series.
- functime forecasters, `naive` against a `linear_model`: a model that does not beat the naive
  forecast has found no trend to extrapolate.

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two. The deeper functime examples (cross-validation, a global model across thousands
of games, the features of the in-game curves) are in [functime.ipynb](./functime.ipynb).

Previous: [Level 2](./Level_2_Over_Time.ipynb). Next: [Level 4](./Level_4_Players.ipynb).

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

import warnings

# flaml, a functime dependency, warns that its optional AutoML extras are not installed.
warnings.filterwarnings("ignore", message="flaml.automl is not available")

import polars_ds as pds
from functime.cross_validation import train_test_split
from functime.forecasting import linear_model, naive
from functime.metrics import mae
from functime.plotting import plot_forecasts

# Registers the `.ts` namespace of the feature extractors on polars expressions.
import functime.feature_extractors  # noqa: F401

# They read every unit_born file, so they are collected once and reused.
players = s2.players(IPC_DIR).collect().lazy()
games = s2.games(IPC_DIR).collect().lazy()
# A month counts with at least this many games, a series with at least this many months.
MIN_GAMES = 10
MIN_MONTHS = 5

## The series

Three monthly series families from Level 2, in one long table `(series, entity, month, value)`:

- `workers_at_8`: the median workers at 8:00, per race.
- `expansion_seconds`: the median game time of the second townhall, per race. Earlier is greedier.
- `minutes`: the median length of the games, per matchup.

`t` is the number of months since the first month of the series: a regression on `t` does not
mind the months without games, a feature extractor on the sequence of values does (see below).
A series needs a few months to say anything, the shorter ones are dropped.

In [2]:
month = pl.col("ext_datetime").dt.truncate("1mo").alias("month")
at_8 = (
    stats.with_columns((s2.game_seconds() // 60).cast(pl.Int32).alias("minute"))
    .filter(pl.col("minute") == 8)
    .group_by("ext_fs_id", "player_id")
    .agg(pl.col("workers_active_count").sort_by("ext_replay_loop").first().alias("value"))
    .join(players.select("ext_fs_id", "player_id", "race", "ext_datetime"), on=["ext_fs_id", "player_id"])
    .group_by(pl.lit("workers_at_8").alias("series"), pl.col("race").alias("entity"), month)
    .agg(pl.len().alias("games"), pl.col("value").median())
)
expansions = (
    unit_born.filter((pl.col("ext_replay_loop") > 0) & pl.col("unit_type_name").is_in(s2.TOWNHALLS))
    .group_by("ext_fs_id", pl.col("control_player_id").alias("player_id"))
    .agg(s2.game_seconds().min().alias("value"))
    .join(players.select("ext_fs_id", "player_id", "race", "ext_datetime"), on=["ext_fs_id", "player_id"])
    .group_by(pl.lit("expansion_seconds").alias("series"), pl.col("race").alias("entity"), month)
    .agg(pl.len().alias("games"), pl.col("value").median())
)
lengths = games.group_by(pl.lit("minutes").alias("series"), pl.col("matchup").alias("entity"), month).agg(
    pl.len().alias("games"), pl.col("minutes").median().alias("value")
)
series = (
    pl.concat([at_8, expansions, lengths])
    .filter((pl.col("games") >= MIN_GAMES) & (pl.col("entity") != "?"))
    .filter(pl.len().over("series", "entity") >= MIN_MONTHS)
    .sort("series", "entity", "month")
    .with_columns(
        ((pl.col("month").dt.year() - pl.col("month").dt.year().min()) * 12 + pl.col("month").dt.month() - pl.col("month").dt.month().first())
        .over("series", "entity")
        .alias("t")
    )
    .collect()
)
series.group_by("series", "entity").agg(pl.len().alias("months"), pl.col("games").sum()).sort("series", "entity")

series,entity,months,games
str,str,u32,u32
"""expansion_seconds""","""P""",73,20616
"""expansion_seconds""","""T""",73,20171
"""expansion_seconds""","""Z""",76,27791
"""minutes""","""PvP""",51,2962
"""minutes""","""PvT""",66,5440
"""minutes""","""PvZ""",66,6123
"""minutes""","""TvT""",47,1840
"""minutes""","""TvZ""",69,6048
"""minutes""","""ZvZ""",57,2999


In [3]:
# The SQL twin of one family, the expansions: the earliest townhall after the start, per player and game.
sql = con.sql(f"""
    WITH expansion AS (
        SELECT ext_fs_id, control_player_id AS player_id, min(game_seconds(ext_replay_loop)) AS value
        FROM unit_born
        WHERE ext_replay_loop > 0 AND unit_type_name IN ({s2.sql_list(s2.TOWNHALLS)})
        GROUP BY ALL
    )
    SELECT 'expansion_seconds' AS series, p.race AS entity, date_trunc('month', p.ext_datetime) AS month, count(*) AS games, median(e.value) AS value
    FROM expansion e
    JOIN players p USING (ext_fs_id, player_id)
    WHERE p.race <> '?'
    GROUP BY p.race, date_trunc('month', p.ext_datetime)
    HAVING count(*) >= {MIN_GAMES}
    QUALIFY count(*) OVER (PARTITION BY entity) >= {MIN_MONTHS}
""")
s2.same(series.filter(pl.col("series") == "expansion_seconds").drop("t"), sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 222 rows, 5 columns'

## The slope of every series, with its uncertainty

`pds.lin_reg_report` fits `value = slope * t + intercept` per group and reports the standard
error and the p-value of each coefficient: a slope whose confidence interval contains zero is not
a trend. The slope is per month. DuckDB's `regr_slope` and `regr_intercept` are the same fit.

In [4]:
trends = (
    series.lazy()
    .group_by("series", "entity")
    .agg(pl.len().alias("months"), pds.lin_reg_report("t", target="value", add_bias=True).alias("fit"))
    .explode("fit")
    .unnest("fit")
    .filter(pl.col("features") == "t")
    .select("series", "entity", "months", pl.col("beta").alias("slope_per_month"), "std_err", pl.col("p>|t|").alias("p_value"), pl.col("0.025").alias("ci_low"), pl.col("0.975").alias("ci_high"), "r2")
    .with_columns((pl.col("p_value") < 0.05).alias("trend"))
    .sort("p_value")
    .collect()
)
trends

series,entity,months,slope_per_month,std_err,p_value,ci_low,ci_high,r2,trend
str,str,u32,f64,f64,f64,f64,f64,f64,bool
"""workers_at_8""","""Z""",76,0.267278,0.036238,1.9486e-10,0.195072,0.339485,0.431253,true
"""minutes""","""ZvZ""",57,-0.036769,0.010773,0.001212,-0.058358,-0.01518,0.189268,true
"""minutes""","""TvT""",47,-0.04867,0.014128,0.001247,-0.077124,-0.020215,0.225529,true
"""workers_at_8""","""T""",73,0.128314,0.041812,0.003042,0.044944,0.211684,0.129207,true
"""expansion_seconds""","""P""",73,-0.169755,0.056248,0.003532,-0.281911,-0.057599,0.125838,true
"""workers_at_8""","""P""",72,0.059143,0.027124,0.032584,0.005047,0.113239,0.076608,true
"""minutes""","""TvZ""",69,-0.019034,0.009832,0.057088,-0.038658,0.00059,0.066703,false
"""expansion_seconds""","""T""",73,0.100367,0.052482,0.059858,-0.004279,0.205014,0.062016,false
"""minutes""","""PvZ""",66,-0.013708,0.008177,0.098517,-0.030043,0.002627,0.056583,false


In [5]:
sql = con.sql("""
    SELECT series, entity, count(*) AS months, regr_slope(value, t) AS slope_per_month, regr_intercept(value, t) AS intercept
    FROM series
    GROUP BY ALL
""")
s2.same(trends.select("series", "entity", "months", "slope_per_month"), sql.select("series, entity, months, slope_per_month"))

'same 12 rows, 4 columns'

## Smoothing: the rolling median against the raw series

Before trusting a slope, look at the series. A rolling mean over three months (`rolling_mean(3)`
per entity, the window `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` in SQL) shows the shape under
the noise. The raw series is drawn thin and gray, the smoothed one in the color of the race.

In [6]:
smoothed = (
    series.lazy()
    .sort("series", "entity", "month")
    .with_columns(pl.col("value").rolling_mean(3, min_samples=1).over("series", "entity").alias("rolling_3"))
    .collect()
)
sql = con.sql("""
    SELECT *, avg(value) OVER (PARTITION BY series, entity ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_3
    FROM series
""")
print(s2.same(smoothed, sql))
# The months without games stay empty in the chart: `upsample` adds them with null values, and
# plotly breaks the line there instead of drawing across the gap.
workers = (
    smoothed.filter(pl.col("series") == "workers_at_8")
    .sort("entity", "month")
    .upsample("month", every="1mo", group_by="entity", maintain_order=True)
    .with_columns(pl.col("entity").forward_fill())
)
fig = px.line(workers, x="month", y="rolling_3", color="entity", color_discrete_map=viz.RACE_COLORS, category_orders={"entity": ["P", "T", "Z"]},
              title="Median workers at 8:00 per race: the 3-month rolling mean over the monthly values", labels={"rolling_3": "workers at 8:00", "entity": "race"})
for race, raw in workers.group_by("entity", maintain_order=True):
    fig.add_scatter(x=raw["month"], y=raw["value"], mode="lines+markers", name="monthly values", legendgroup="monthly", line=dict(color=viz.MUTED, width=1), marker=dict(size=5, color=viz.MUTED), showlegend=race[0] == "P")
viz.show(fig, "level_3_workers_trend")

same 799 rows, 7 columns


## The feature extractors of functime

One number per series, as polars expressions in the `.ts` namespace. They work on the sequence of
values, so the months without games are filled first with the previous value (`upsample` and
`forward_fill`): a regular series is what the forecasters below need too.

- `linear_trend`: the slope per step (the same as `regr_slope` on a regular series).
- `mean_change`: the average change from one month to the next.
- `absolute_sum_of_changes`: how much the series moves in total, noise has a large value.
- `longest_streak_above_mean`: the longest run of months above the average of the series.

In [7]:
regular = (
    series.lazy()
    .select("series", "entity", "month", "value")
    .sort("series", "entity", "month")
    .collect()
    .upsample("month", every="1mo", group_by=["series", "entity"], maintain_order=True)
    .with_columns(pl.col("series", "entity", "value").forward_fill())
)
features = (
    regular.lazy()
    .group_by("series", "entity")
    .agg(
        pl.len().alias("months"),
        pl.col("value").ts.linear_trend().struct.field("slope").alias("slope_per_month"),
        pl.col("value").ts.mean_change().alias("mean_change"),
        pl.col("value").ts.absolute_sum_of_changes().alias("absolute_changes"),
        pl.col("value").ts.longest_streak_above_mean().alias("longest_streak_above_mean"),
    )
    .sort("series", "entity")
    .collect()
)
features

series,entity,months,slope_per_month,mean_change,absolute_changes,longest_streak_above_mean
str,str,u32,f64,f64,f64,u32
"""expansion_seconds""","""P""",85,-0.091228,0.267857,629.375,7
"""expansion_seconds""","""T""",85,0.150498,0.20169,599.575893,7
"""expansion_seconds""","""Z""",87,-0.007049,0.012199,169.799107,12
"""minutes""","""PvP""",82,0.007972,0.014141,72.257068,8
"""minutes""","""PvT""",85,-0.011437,-0.030475,82.702753,12
"""minutes""","""PvZ""",85,-0.001987,0.026728,76.462426,14
"""minutes""","""TvT""",81,-0.025773,-0.032771,80.28311,11
"""minutes""","""TvZ""",85,0.000365,0.049904,114.551339,12
"""minutes""","""ZvZ""",85,-0.056893,0.014465,78.482143,24


In [8]:
sql = con.sql("""
    WITH indexed AS (SELECT *, row_number() OVER (PARTITION BY series, entity ORDER BY month) - 1 AS t FROM regular)
    SELECT series, entity, count(*) AS months, regr_slope(value, t) AS slope_per_month
    FROM indexed
    GROUP BY ALL
""")
s2.same(features.select("series", "entity", "months", "slope_per_month"), sql)

'same 12 rows, 4 columns'

## Forecasting: does a model beat the naive forecast?

functime forecasts panel data, a long frame of `(entity, time, value)`. The last three months of
each series are kept for the test, `naive` repeats the last value and `linear_model(lags=3)`
regresses on the previous three months, one model for all the entities of the series. The mean
absolute error on the test months says which is better: when `naive` wins there is nothing to
extrapolate, which is a finding too. The entities need the test months plus the lags of
training, the short ones are left out.

In [9]:
FH, LAGS = 3, 3
long_enough = regular.filter(pl.len().over("series", "entity") >= FH + 2 * LAGS)
rows = []
forecasts = {}
for name, panel in long_enough.group_by("series", maintain_order=True):
    y = panel.select("entity", "month", "value")
    y_train, y_test = train_test_split(test_size=FH, eager=True)(y)
    for model_name, model in {"naive": naive(freq="1mo"), "linear 3 lags": linear_model(freq="1mo", lags=LAGS)}.items():
        y_pred = model(y=y_train, fh=FH)
        forecasts[(name[0], model_name)] = y_pred
        rows.append({"series": name[0], "model": model_name, "mae": mae(y_true=y_test, y_pred=y_pred)["mae"].mean()})
pl.DataFrame(rows).pivot(on="model", index="series", values="mae")

/home/seb/s2-polars-levels/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))
/home/seb/s2-polars-levels/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))
/home/seb/s2-polars-levels/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially 

series,naive,linear 3 lags
str,f64,f64
"""expansion_seconds""",16.428571,21.022009
"""minutes""",0.93564,1.421775
"""workers_at_8""",2.111111,5.988772


In [10]:
fig = plot_forecasts(
    y_true=long_enough.filter(pl.col("series") == "workers_at_8").select("entity", "month", "value"),
    y_pred=forecasts[("workers_at_8", "linear 3 lags")],
    num_cols=3,
    layout_kwargs={"height": 400, "width": 1000, "title": "Workers at 8:00 per race: the last 3 months forecast by the linear model"},
)
viz.show(fig, "level_3_forecast")

## What to take from Level 3

- A slope without its standard error is not a trend. `pds.lin_reg_report` gives both in one
  expression, `regr_slope` is its SQL twin; filter the months with few games first.
- Smooth to look (rolling mean), regularize to compute (upsample and fill), and say which one
  the chart shows.
- functime's `.ts` features summarize a series in one number and its forecasters tell whether
  the past months predict the next ones; `naive` is the baseline to beat.

Next: [Level 4](./Level_4_Players.ipynb) compares the players, [Level 5](./Level_5_Maps.ipynb)
the maps.